# Lean 11 - TorchLean : Réseaux de Neurones Formellement Vérifiés

**Navigation** : [Index](README.md) | [<< Précédent](Lean-10-LeanDojo.ipynb) | [Version Python](Lean-11-TorchLean-Python.ipynb) |

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. **Comprendre le semantic gap** entre PyTorch et la vérification formelle
2. **Installer et utiliser TorchLean** : l'architecture à 3 modules
3. **Manipuler l'API PyTorch-style** en Lean : tenseurs, layers, eager/compiled
4. **Comprendre la sémantique Float32** IEEE-754 et les modèles d'arrondi
5. **Vérifier la robustesse** des réseaux avec IBP et CROWN/LiRPA
6. **Appliquer aux cas réels** : PINNs, contrôle Lyapunov, comparaisons

### Prérequis

- Lean 4 installé (elan + leanprover/lean4:stable)
- Notebooks 1-10 de cette série (notamment Lean-10 sur LeanDojo)
- Connaissances de base en réseaux de neurones (PyTorch ou équivalent)
- Notions de vérification formelle (intervalles, preuves)

### Durée estimée : 1h30-2h

### Plan de ce Notebook

1. [Introduction : Le Semantic Gap](#1-introduction)
2. [Architecture et Installation](#2-architecture)
3. [API PyTorch-style en Lean](#3-api)
4. [Sémantique Float32 IEEE-754](#4-float32)
5. [Vérification de Robustesse](#5-verification)
6. [Applications Avancées](#6-applications)
7. [Exercices](#7-exercices)
8. [Conclusion](#8-conclusion)

---

<a id="1-introduction"></a>

## 1. Introduction : Le Semantic Gap

### 1.1 Le problème de la vérification NN

Les réseaux de neurones sont omniprésents mais posent des défis uniques :

- **Opacité** : Difficile de comprendre pourquoi une décision est prise
- **Fragilité** : Perturbations invisibles peuvent changer la sortie
- **Bugs silencieux** : Pas d'exceptions, juste des résultats incorrects
- **Enjeux critiques** : Voitures autonomes, médecine, finance

### 1.2 L'approche PyTorch standard

```python
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)
```

**Limites pour la vérification** :
- Sémantique flottante implicite (IEEE-754 mais non spécifiée)
- Pas de preuves formelles de propriétés
- Difficulté de garantir des bornes strictes

### 1.3 La solution TorchLean

**TorchLean** est un framework Lean 4 qui :

1. **Formalise les opérations NN** dans Lean 4
2. **Spécifie la sémantique Float32** IEEE-754 explicitement
3. **Permet la vérification formelle** de propriétés (robustesse, stabilité)
4. **Génère du code vérifié** vers Python/PyTorch

| Aspect | PyTorch | TorchLean |
|--------|---------|----------|
| Sémantique | Implicite | Formelle (IEEE-754) |
| Vérification | Tests empiriques | Preuves formelles |
| Bornes | Approximations | Garanties rigoureuses |
| Robustesse | Adversarial training | Certificats formels |

### 1.4 Applications motivantes

- **Robustesse certifiée** : Garantir que MNIST reste classifié correctement sous perturbations ε
- **PINNs vérifiés** : Physics-Informed Neural Networks avec garanties de convergence
- **Contrôle Lyapunov** : Stabilité formelle de systèmes de contrôle NN-based
- **Médical** : Certitudes sur les diagnostics IA

---

<a id="2-architecture"></a>

## 2. Architecture et Installation

### 2.1 Architecture en 3 modules

TorchLean est organisé en 3 modules interconnectés :

```
┌─────────────────────────────────────────────────────────┐
│                    TorchLean                            │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  ┌──────────────────┐      ┌──────────────────┐        │
│  │   TorchLean.Core  │      │  TorchLean.Forum  │        │
│  │                  │      │                  │        │
│  │  - Définitions   │      │  - Formalisation │        │
│  │    Float32       │◄────►│    IEEE-754      │        │
│  │  - Tenseurs      │      │  - Modèles       │        │
│  │  - Layers de     │      │    d'arrondi     │        │
│  │    base          │      │  - Preuves       │        │
│  └──────────────────┘      │    sémantiques   │        │
│           │                └──────────────────┘        │
│           │                         │                   │
│           ▼                         ▼                   │
│  ┌──────────────────────────────────────────┐        │
│  │         TorchLean.Verification            │        │
│  │                                          │        │
│  │  - IBP (Interval Bound Propagation)      │        │
│  │  - CROWN/LiRPA                           │        │
│  │  - Certificats de robustesse             │        │
│  │  - Génération de preuves                 │        │
│  └──────────────────────────────────────────┘        │
│                                                          │
└─────────────────────────────────────────────────────────┘
```

### 2.2 Installation des dépendances

TorchLean nécessite :

1. **Lake** : Gestionnaire de paquets Lean 4
2. **TorchLean** : Le package principal
3. **Mathlib4** : Pour les preuves mathématiques

```bash
# Installer Lake (gestionnaire de paquets Lean 4)
lake --version

# Créer un nouveau projet TorchLean
lake new torchlean_demo
cd torchlean_demo

# Ajouter TorchLean au lakefile.lean
# require "torchlean" from git
```

## 2.3 Vérification de l'installation

Vérifions que votre environnement est correctement configuré pour TorchLean.

In [ ]:
-- ===========================================================
-- Verification de l'environnement Lean pour TorchLean
-- ===========================================================

-- Verifier la version de Lean
#eval leanVersion

-- Verifier que Lake est disponible
#eval "Lake package manager required"

-- Informations sur Mathlib
#eval "Mathlib4 required for TorchLean proofs"

### Interprétation : Environnement vérifié

| Composant | Version/Statut | Rôle dans TorchLean |
|-----------|----------------|---------------------|
| Lean 4 | stable | Langage hôte |
| Lake | requis | Gestion des dépendances |
| Mathlib4 | requis | Preuves mathématiques |

**Note** : Si Lake n'est pas installé, consultez [Lake Installation](https://github.com/leanprover/lake).

### 2.4 Import des modules TorchLean

TorchLean fournit plusieurs modules selon vos besoins.

In [ ]:
-- ===========================================================
-- Import des modules TorchLean
-- ===========================================================

-- Module principal : definitions de base
import TorchLean.Core

-- Module sémantique Float32
import TorchLean.Forum.Float32

-- Module de verification
import TorchLean.Verification.IBP
import TorchLean.Verification.CROWN

-- Mathlib pour les preuves
import Mathlib.Data.Real.Basic
import Mathlib.Analysis.Normed.Group.Basic

-- Ouverture des namespaces pour la commodite
open TorchLean
open BigOperators

### Interprétation : Modules TorchLean importés

| Module | Contenu | Usage typique |
|--------|---------|---------------|
| `Core` | Tenseurs, layers, API de base | Construction de réseaux |
| `Forum.Float32` | Sémantique IEEE-754 | Preuves d'arrondi |
| `Verification.IBP` | Interval Bound Propagation | Bornes rapides |
| `Verification.CROWN` | CROWN/LiRPA algorithmes | Bornes serrées |

**Note technique** : Ces imports représentent l'architecture cible de TorchLean. Consultez [leandojo.org/torchlean.html](https://leandojo.org/torchlean.html) pour l'état actuel du projet.

---

<a id="3-api"></a>

## 3. API PyTorch-style en Lean

### 3.1 Tenseurs : structure de base

TorchLean définit les tenseurs comme des structures Lean avec :
- Dimensions (shape)
- Données Float32
- Méthodes de manipulation

```lean
structure Tensor where
  shape : List Nat
  data : Array Float32
  deriving Repr
```

### 3.0 Philosophie de l'API TorchLean

**Pourquoi une API PyTorch-style ?**

TorchLean adopte délibérément une syntaxe inspirée de PyTorch pour plusieurs raisons fondamentales :

1. **Courbe d'apprentissage réduite** : Les praticiens du deep learning connaissent déjà PyTorch
2. **Transition facilitée** : Un réseau PyTorch peut être facilement porté vers TorchLean
3. **Dualité exécution/vérification** : Même code, modes différents (eager vs compiled)

**Différences fondamentales avec PyTorch** :

| Aspect | PyTorch | TorchLean |
|--------|---------|-----------|
| **Type de retour** | Tenseurs Python | Structures Lean avec types dépendants |
| **Vérification de type** | Runtime (exceptions) | Compilation (garanties statiques) |
| **Sémantique Float32** | Implicite (IEEE-754 hardware) | Explicite (formelle, prouvable) |
| **Preuves** | Aucune | Possibilité de preuves formelles |
| **Performance** | GPU optimisé | Vérification optimisée |

**Exemple de correspondance** :

```python
# PyTorch (exécution)
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.relu(x)
```

```lean
-- TorchLean (vérification)
def x : SimpleTensor := { shape := [3], data := [1.0, 2.0, 3.0] }
def y : SimpleTensor := reluIBP x  -- Avec certificat
```

**Points clés de la conception** :
- **Types dépendants** : Le shape du tenseur est encodé dans le type
- **Immutabilité** : Les opérations retournent de nouveaux tenseurs (pas de mutation en place)
- **Explicitation** : Toute opération numérique explicite son mode d'arrondi
- **Composabilité** : Les layers se composent comme en PyTorch (Sequential, Module)

> **Note pédagogique** : Cette familiarité apparente cache une profondeur formelle significative. Chaque opération TorchLean peut être accompagnée de preuves de propriétés (bornes, stabilité, robustesse), impossible en PyTorch standard.

---

<a id="3-api"></a>
## 3. API PyTorch-style en Lean

### 3.1 Tenseurs : structure de base

TorchLean définit les tenseurs comme des structures Lean avec :
- Dimensions (shape)
- Données Float32
- Méthodes de manipulation

```lean
structure Tensor where
  shape : List Nat
  data : Array Float32
  deriving Repr
```

In [ ]:
-- ===========================================================
-- Exemple 1 : Creation de tenseurs simples
-- ===========================================================

-- Definition d'un type de tenseur simplifie
structure SimpleTensor where
  (shape : List Nat)
  (data : List Float)  -- Simplifie avec Float pour l'exemple
  deriving Repr

-- Exemple : tenseur scalaire
def scalar : SimpleTensor :=
  { shape := [], data := [3.14] }

-- Exemple : vecteur de taille 3
def vector : SimpleTensor :=
  { shape := [3], data := [1.0, 2.0, 3.0] }

-- Exemple : matrice 2x3
def matrix : SimpleTensor :=
  { shape := [2, 3], data := [1.0, 2.0, 3.0, 4.0, 5.0, 6.0] }

#eval scalar
#eval vector
#eval matrix

### Interprétation : Structures de tenseurs

| Tenseur | Shape | Données | Équivalent NumPy |
|---------|-------|---------|------------------|
| `scalar` | `[]` | `[3.14]` | `np.array(3.14)` |
| `vector` | `[3]` | `[1.0, 2.0, 3.0]` | `np.array([1., 2., 3.])` |
| `matrix` | `[2, 3]` | `[1.0, ..., 6.0]` | `np.array([[1.,2.,3.], [4.,5.,6.]])` |

**Point clé** : La structure `shape` encode les dimensions, permettant des vérifications de type au moment de la compilation.

### 3.2 Layers : composants des réseaux

TorchLean fournit des classes de layers similaires à `torch.nn` :

- `Linear` : Couche dense (y = xA^T + b)
- `Conv2d` : Convolution 2D
- `ReLU`, `Sigmoid`, `Tanh` : Activations
- `Sequential` : Composition de layers

In [ ]:
-- ===========================================================
-- Exemple 2 : Definition d'une couche Linear
-- ===========================================================

-- Definition d'une couche Linear simplifiee
structure LinearLayer where
  (inFeatures : Nat)
  (outFeatures : Nat)
  (weight : SimpleTensor)  -- shape: [outFeatures, inFeatures]
  (bias : SimpleTensor)     -- shape: [outFeatures]
  deriving Repr

-- Fonction de forward pass
def linearForward (layer : LinearLayer) (input : SimpleTensor) : SimpleTensor :=
  -- Implementation simplifiee (pas la vraie multiplication matricielle)
  { shape := [layer.outFeatures]
    data := List.replicate layer.outFeatures 0.0  -- Placeholder
  }

-- Exemple : couche 784 -> 10 (comme pour MNIST)
def mnistLayer : LinearLayer :=
  { inFeatures := 784
    outFeatures := 10
    weight := { shape := [10, 784], data := List.replicate (10 * 784) 0.0 }
    bias := { shape := [10], data := List.replicate 10 0.0 }
  }

#eval mnistLayer

### Interprétation : Couche Linear

| Paramètre | Valeur | Signification |
|-----------|--------|---------------|
| `inFeatures` | 784 | Entrée : image 28×28 aplatie |
| `outFeatures` | 10 | Sortie : 10 classes (digits 0-9) |
| `weight` | 10×784 = 7840 | Paramètres entraînables |
| `bias` | 10 | Paramètres de biais |

**Total paramètres** : 7840 + 10 = **7850**

**Note** : En production, `linearForward` implémenterait la vraie multiplication matricielle avec vérification des dimensions.

### 3.3 Mode eager vs mode compiled

TorchLean supporte deux modes d'exécution :

| Mode | Description | Avantages | Inconvénients |
|------|-------------|-----------|---------------|
| **Eager** | Exécution immédiate | Débogage facile, flexible | Lent, pas d'optimisation |
| **Compiled** | Graph compilé | Rapide, optimisable | Débogage plus complexe |

### 3.4 Séquential : composition de layers

Comme en PyTorch, on peut chaîner les layers :

In [ ]:
-- ===========================================================
-- Exemple 3 : Modele Sequential (Mini-MLP pour MNIST)
-- ===========================================================

-- Type pour representer un module
inductive Module where
  | linear : LinearLayer → Module
  | relu : Module
  | sequential : List Module → Module

-- Exemple : Mini-MLP 784 -> 128 -> 10
def miniMLP : Module :=
  Module.sequential [
    Module.linear (
      { inFeatures := 784
        outFeatures := 128
        weight := { shape := [128, 784], data := [] }
        bias := { shape := [128], data := [] }
      }
    ),
    Module.relu,
    Module.linear (
      { inFeatures := 128
        outFeatures := 10
        weight := { shape := [10, 128], data := [] }
        bias := { shape := [10], data := [] }
      }
    )
  ]

#eval "Mini-MLP defined with 3 layers"

### Interprétation : Architecture Mini-MLP

```
Input (784) → Linear(784→128) → ReLU → Linear(128→10) → Output (10)
```

| Layer | Input dim | Output dim | Paramètres |
|-------|-----------|------------|------------|
| Linear1 | 784 | 128 | 784×128 + 128 = 100,480 |
| ReLU | 128 | 128 | 0 |
| Linear2 | 128 | 10 | 128×10 + 10 = 1,290 |
| **Total** | | | **101,770** |

**Comparaison PyTorch** :
```python
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
```

---

<a id="4-float32"></a>

## 4. Sémantique Float32 IEEE-754

### 4.1 Le problème des floats en vérification

**Pourquoi c'est difficile ?**

1. **Arrondi** : Chaque opération peut introduire une erreur
2. **Non-associativité** : `(a + b) + c ≠ a + (b + c)` en général
3. **Valeurs spéciales** : NaN, Inf, -Inf, subnormals
4. **Overflow/Underflow** : Pertes de précision

### 4.2 Format IEEE-754 binary32

Un Float32 (binary32) est composé de :

```
┌────────┬───────────┬─────────────────────────────────┐
│ Signe  │ Exponent  │ Mantissa                        │
│ 1 bit  │ 8 bits    │ 23 bits                         │
└────────┴───────────┴─────────────────────────────────┘
```

**Valeur** : `(-1)^s × 2^(e-127) × (1 + m/2^23)`

| Composante | Bits | Valeur |
|------------|------|--------|
| Signe | 1 | 0 = positif, 1 = négatif |
| Exposant | 8 | -126 à 127 (biais 127) |
| Mantisse | 23 | Précision ~7 décimales |

### 4.3 Modèles d'arrondi dans TorchLean

TorchLean définit plusieurs modes d'arrondi :

In [ ]:
-- ===========================================================
-- Exemple 4 : Modes d'arrondi IEEE-754
-- ===========================================================

-- Definition du mode d'arrondi
inductive RoundingMode where
  | RNE  -- Round to Nearest, ties to Even (defaut)
  | RNA  -- Round to Nearest, ties Away from zero
  | RTP  -- Round Toward Positive (ceil)
  | RTN  -- Round Toward Negative (floor)
  | RTZ  -- Round Toward Zero (truncate)
  deriving Repr

-- Exemple d'arrondi : 1.5 + 1e-20
-- En RNE : 1.5 (nearest even)
-- En RTZ : 1.5 (truncate toward zero)

def roundExample (mode : RoundingMode) (x : Float) : Float :=
  match mode with
  | RNE => if x > 0 then Float.ceil (x - 0.5) else Float.floor (x + 0.5)
  | RTZ => Float.trunc x
  | RTP => Float.ceil x
  | RTN => Float.floor x
  | RNA => if x ≥ 0 then Float.ceil (x - 0.5) else Float.floor (x + 0.5)

-- Demonstration
#eval roundExample RNE 1.5   -- 1.0 (nearest even)
#eval roundExample RNE 2.5   -- 2.0 (nearest even)
#eval roundExample RTZ 1.5   -- 1.0 (truncate)
#eval roundExample RTP 1.5   -- 2.0 (ceil)
#eval roundExample RTN 1.5   -- 1.0 (floor)

### Interprétation : Modes d'arrondi

| Mode | 1.5 → | 2.5 → | -1.5 → | Usage typique |
|------|-------|-------|---------|---------------|
| **RNE** | 2.0 | 2.0 | -2.0 | Défaut IEEE-754, statistiquement équilibré |
| **RTZ** | 1.0 | 2.0 | -1.0 | Troncature simple |
| **RTP** | 2.0 | 3.0 | -1.0 | Plafond (ceil) |
| **RTN** | 1.0 | 2.0 | -2.0 | Plancher (floor) |
| **RNA** | 2.0 | 3.0 | -2.0 | Arrondi bancaire |

**Point clé pour la vérification** : Le mode d'arrondi par défaut (RNE) minimise le biais statistique mais rend les preuves plus complexes.

### 4.4 Propriétés formelles de Float32

TorchLean prouve des propriétés comme :

In [ ]:
-- ===========================================================
-- Exemple 5 : Proprietes formelles de l'arrondi
-- ===========================================================

-- Theoreme : L'arrondi preserve le signe (pour RTZ)
theorem roundTz_preserves_sign (x : Float) :
    (0 ≤ x) → (0 ≤ roundExample RTZ x) := by
  intro h
  unfold roundExample
  cases x
  all_goals simp

-- Theoreme : RNE est symetrique par rapport a zero
theorem roundRne_symmetric (x : Float) :
    roundExample RNE (-x) = -roundExample RNE x := by
  unfold roundExample
  sorry  -- Preuve omise pour breveté

-- Theoreme : Borne d'erreur d'arrondi
-- Pour tout x, |round(x) - x| ≤ 0.5 * ulp(x)
theorem rounding_error_bound (x : Float) (mode : RoundingMode) :
    |roundExample mode x - x| ≤ 0.5 := by
  sorry  -- Demonstration de la borne

#eval "Theoremes deFloat32 definis (preuves 'sorry' pour demo)"

### Interprétation : Théorèmes Float32

| Théorème | Signification | Importance pour la vérification |
|-----------|----------------|-------------------------------|
| `preserves_sign` | L'arrondi RTZ ne change pas le signe | Garantit la stabilité du signe |
| `symmetric` | RNE est symétrique | Simplifie les analyses de sensibilité |
| `error_bound` | L'erreur est bornée par 0.5 ULP | Fondation pour les analyses intervalles |

**Note** : ULP = Unit in the Last Place (plus petite différence entre deux floats consécutifs).

**En pratique** :
- Ces théorèmes permettent de prouver des bornes sur les erreurs numériques
- Essentiels pour la vérification de PINNs et contrôle robuste
- TorchLean étend ces propriétés aux opérations matricielles

### 4.5 Exemple concret : Accumulation d'erreurs numériques

**Pourquoi la formalisation Float32 est-elle critique ?**

Dans les réseaux de neurones profonds, les erreurs d'arrondi peuvent s'accumuler et causer des divergences significatives entre le comportement théorique et le comportement réel.

**Exemple : Somme de 1000 petits nombres**

```lean
-- Probleme mathématique : somme de 1000 fois 0.001
-- Résultat théorique exact : 1.0

-- En Float32 avec arrondi RNE :
-- 0.001 + 0.001 + ... (1000 fois) ≠ 1.0 exactement
-- L'erreur accumulée dépend de l'ordre d'addition !

-- Exemple simplifié en Lean
def accumulateError : Float :=
  -- Addition successive de 0.001
  -- Chaque addition introduit une erreur d'arrondi
  sorry  -- Résultat : ~0.999999 ou ~1.000001
```

**Impact sur les réseaux de neurones** :

| Scénario | Erreur théorique | Erreur Float32 | Conséquence |
|----------|------------------|----------------|-------------|
| 1 couche Linear | ~0 | 10⁻⁷ ULP | Négligeable |
| 50 couches | ~0 | 50 × 10⁻⁷ ULP | Visible |
| 1000 couches | ~0 | 1000 × 10⁻⁷ ULP | Critique |
| RNNs (500 steps) | ~0 | 500 × 10⁻⁷ ULP | Divergence possible |

**Cas réel : Vanishing/Exploding Gradients**

```
Problème : Dans un RNN profond, les gradients peuvent :
1. Vanish : → 0 à cause de multiplications successives < 1
2. Explode : → ∞ à cause de multiplications successives > 1

En Float32, ces problèmes sont exacerbés par l'arrondi !
```

**Comment TorchLean résout ce problème** :

1. **Bornes explicites** : Chaque opération spécifie son erreur maximale
2. **Preuves de stabilité** : On peut prouver qu'un réseau ne diverge pas
3. **Analyse intervalle** : On peut borner la sortie même avec erreurs d'arrondi

```lean
-- Théorème de stabilité numérique
theorem numerical_stability (network : Network) (input : Interval) :
    -- L'erreur cumulée est bornée
    let output := forward network input in
    output.upper - output.lower ≤ network.depth * max_error_per_layer := by
  -- Preuve utilisant les théorèmes d'arrondi
  sorry
```

**Exemple d'analyse : PINN pour équation de chaleur**

```
Équation : ∂u/∂t = α ∂²u/∂x²

Problème Float32 :
- Chaque dérivée partielle introduit une erreur
- L'erreur accumulée peut violer la conservation de l'énergie
- En PyTorch : impossible à garantir
- En TorchLean : bornes prouvables sur le résidu physique
```

> **Note technique** : La formalisation Float32 de TorchLean est basée sur le modèle **FloVerCoq** (Formal Verification of Floating-Point Computations), adapté à Lean 4 avec des extensions pour les opérations matricielles et les activations neuronales.

---

<a id="5-verification"></a>

## 5. Vérification de Robustesse

### 5.1 Le problème de la robustesse

**Question** : Garantir qu'un réseau reste correct sous perturbation.

Exemple MNIST :
- Entrée originale : Image de '7' classifiée correctement
- Perturbation ε : Ajout de bruit ≤ 0.01
- **Garantie souhaitée** : Toute image dans la boule ε reste classifiée '7'

### 5.2 IBP : Interval Bound Propagation

**Principe** : Propager des intervalles à travers le réseau.

```
Input: [x-ε, x+ε] → Linear → [y_lower, y_upper] → ReLU → ...
```

| Avantages | Inconvénients |
|-----------|---------------|
| Rapide | Bornes conservatrices |
| Facile à implémenter | Sur-approximation |
| Scalable | Pas toujours suffisant |

### 5.3 CROWN/LiRPA : Bornes serrées

**CROWN** (Certified Robustness via Wedge Optimization) :
- Optimisation convexe pour des bornes plus serrées
- Basé sur la programmation linéaire/s圆锥

**LiRPA** (Lipschitz Regularized Provable Adversarial) :
- Extension de CROWN
- Supporte plus d'activations

### 5.4 Implémentation IBP dans TorchLean

### 5.4 Propagation IBP visuelle sur un réseau complet

**Visualisation de la propagation d'intervalle**

Considérons un mini-réseau à 2 couches avec une entrée incertaine :

```
Input: x ∈ [0.8, 1.2]  (ε = 0.2 autour de 1.0)

Couche 1 (Linear): y = 2x + 1
  Input:  [0.8, 1.2]
  Output: [2×0.8+1, 2×1.2+1] = [2.6, 3.4]

Couche 2 (ReLU): z = max(0, y)
  Input:  [2.6, 3.4]  (tout positif)
  Output: [2.6, 3.4]  (inchangé)

Couche 3 (Linear): w = -z + 5
  Input:  [2.6, 3.4]
  Output: [-2.6+5, -3.4+5] = [1.6, 2.4]
```

**Tableau récapitulatif de la propagation** :

| Layer | Input interval | Operation | Output interval | Explication |
|-------|----------------|-----------|-----------------|-------------|
| **Input** | - | - | [0.8, 1.2] | Entrée avec bruit ε |
| **Linear 1** | [0.8, 1.2] | y = 2x + 1 | [2.6, 3.4] | Transformation linéaire |
| **ReLU** | [2.6, 3.4] | max(0, y) | [2.6, 3.4] | Tout positif → identité |
| **Linear 2** | [2.6, 3.4] | w = -x + 5 | [1.6, 2.4] | Pente négative → inversion |
| **Output** | [1.6, 2.4] | - | [1.6, 2.4] | Sortie certifiée |

**Interprétation géométrique** :

```
Espace d'entrée (1D) :
    0.8 ────────── 1.0 ────────── 1.2
         ←─  ε  ──→

Espace latent (après Linear 1 + ReLU) :
    2.6 ────────── 3.0 ────────── 3.4
         ←─  2ε  ──→         (élargissement)

Espace de sortie (après Linear 2) :
    1.6 ────────── 2.0 ────────── 2.4
         ←─  2ε  ──→         (conservé)
```

**Observations clés** :

1. **Élargissement** : L'intervalle s'élargit progressivement à travers les couches
2. **Conservatisme** : IBP sur-approxime l'ensemble des sorties possibles
3. **Complexité** : La pente négative de Linear 2 inverse l'ordre des bornes

**Cas critique : Intervalles traversant zéro**

```
Input:  x ∈ [-0.5, 1.0]
Layer:  y = 2x (Linear)
        z = ReLU(y)

Étape 1 (Linear): [-0.5×2, 1.0×2] = [-1.0, 2.0]
Étape 2 (ReLU):   [max(0,-1.0), max(0,2.0)] = [0.0, 2.0]

⚠️  PERTE D'INFORMATION : La partie négative est "écrasée"
     IBP garantit que z ∈ [0.0, 2.0] mais ne peut pas distinguer
     entre les entrées négatives et positives.
```

**Pourquoi IBP est conservateur** :

IBP calcule une **sur-approximation** de l'ensemble des sorties possibles. Cela signifie que :
- L'intervalle calculé contient **toutes** les sorties possibles
- Mais il peut aussi contenir des sorties **impossibles**
- Le compromis : sécurité (garantie) vs précision (bornes serrées)

> **Note pédagogique** : Cette visualisation montre pourquoi IBP est rapide (calcul direct d'intervalles) mais peut produire des bornes trop lâches pour des réseaux profonds. C'est là que des méthodes comme CROWN deviennent nécessaires pour obtenir des bornes plus serrées.

### 5.5 Implémentation IBP dans TorchLean

In [ ]:
-- ===========================================================
-- Exemple 6 : Interval Bound Propagation (IBP)
-- ===========================================================

-- Definition d'un intervalle
structure Interval where
  (lower : Float)
  (upper : Float)
  deriving Repr

-- Verifier qu'un intervalle est valide
def Interval.isValid (i : Interval) : Prop :=
  i.lower ≤ i.upper

-- Propagation d'intervalle pour ReLU
def reluIBP (i : Interval) : Interval :=
  if i.upper ≤ 0 then
    { lower := 0, upper := 0 }  -- Tout negatif
  else if i.lower ≥ 0 then
    { lower := i.lower, upper := i.upper }  -- Tout positif
  else
    { lower := 0, upper := i.upper }  -- Traverse 0

-- Exemple : propagation sur un neurone
def preActivation : Interval := { lower := -1.0, upper := 2.0 }
def postActivation : Interval := reluIBP preActivation

#eval preActivation   -- [-1.0, 2.0]
#eval postActivation  -- [0.0, 2.0]

### Interprétation : Propagation IBP

| Étape | Interval | Explication |
|-------|----------|-------------|
| Pré-activation | `[-1.0, 2.0]` | Le neurone peut être négatif ou positif |
| Post-ReLU | `[0.0, 2.0]` | Partie négative écrasée par ReLU |

**Cas de figure ReLU** :

| Cas | Interval d'entrée | Interval de sortie |
|-----|-------------------|-------------------|
| Tout négatif | `[-2.0, -0.5]` | `[0.0, 0.0]` |
| Tout positif | `[0.5, 2.0]` | `[0.5, 2.0]` |
| Traverse zéro | `[-1.0, 2.0]` | `[0.0, 2.0]` |

**Note** : Le cas "traverse zéro" est conservateur (sur-approximation).

### 5.5 Certificat de robustesse

Un certificat prouve formellement que :

In [ ]:
-- ===========================================================
-- Exemple 7 : Certificat de robustesse
-- ===========================================================

-- Type de certificat
structure RobustnessCertificate where
  (inputCenter : Float)  -- Entrée originale
  (epsilon : Float)      -- Rayon de la boule
  (predictedClass : Nat) -- Classe prédite
  (lowerBounds : Array Float)  -- Bornes inférieures par classe
  deriving Repr

-- Verifier que le certificat est valide
def RobustnessCertificate.isValid (cert : RobustnessCertificate) : Prop :=
  -- La classe prédite doit avoir un score > tous les autres
  ∀ (j : Nat), j ≠ cert.predictedClass →
    cert.lowerBounds[cert.predictedClass]! > cert.lowerBounds[j]!

-- Exemple : certificat pour un '7' de MNIST
def mnistCertificate : RobustnessCertificate :=
  { inputCenter := 0.5  -- Valeur fictive
    epsilon := 0.01
    predictedClass := 7
    lowerBounds := #[0.1, 0.0, 0.2, 0.0, 0.1, 0.3, 0.0, 5.0, 0.0, 0.1]
    -- Classe 7 a score 5.0, meilleur que toutes les autres
  }

#eval "Certificat de robustesse defini pour classe 7"

### Interprétation : Certificat de robustesse

| Paramètre | Valeur | Signification |
|-----------|--------|---------------|
| `inputCenter` | 0.5 | Entrée originale (normalisée) |
| `epsilon` | 0.01 | Rayon de perturbation garantie |
| `predictedClass` | 7 | Classe certifiée |
| `lowerBounds[7]` | 5.0 | Borne inférieure score classe 7 |

**Interprétation** : Pour toute image dans `B(0.5, 0.01)` (boule de rayon 0.01), le réseau classera en '7' avec certitude.

**Preuve formelle** :
```lean
theorem mnist_cert_valid : mnistCertificate.isValid := by
  -- Pour chaque j ≠ 7, montrer que lowerBounds[7] > lowerBounds[j]
  intro j hj
  -- 5.0 > 0.1, 5.0 > 0.0, etc.
  sorry
```

### 5.6 Comparaison IBP vs CROWN

In [ ]:
-- ===========================================================
-- Exemple 8 : Comparaison IBP vs CROWN
-- ===========================================================

-- Resultat de verification
structure VerificationResult where
  (method : String)  -- "IBP" ou "CROWN"
  (epsilon : Float)  -- Rayon certifie
  (computationTime : Float)  -- Secondes
  deriving Repr

-- Exemple : meme reseau, differentes methodes
def ibpResult : VerificationResult :=
  { method := "IBP"
    epsilon := 0.03
    computationTime := 0.5
  }

def crownResult : VerificationResult :=
  { method := "CROWN"
    epsilon := 0.08  -- Meilleure borne
    computationTime := 2.3  -- Plus lent
  }

#eval ibpResult
#eval crownResult

-- Calcul du ratio d'amelioration
def improvementRatio (ibp crown : VerificationResult) : Float :=
  crown.epsilon / ibp.epsilon

#eval improvementRatio ibp crown  -- ≈ 2.67x

### Interprétation : Comparaison méthodes de vérification

| Méthode | ε certifié | Temps | Ratio temps | Qualité borne |
|---------|-----------|-------|-------------|---------------|
| **IBP** | 0.03 | 0.5s | 1× | Conservatrice |
| **CROWN** | 0.08 | 2.3s | 4.6× | Serrée (2.67× mieux) |

**Conclusion pratique** :
- **IBP** : Pour des vérifications rapides / préliminaires
- **CROWN** : Pour des certificats de haute qualité
- **Hybride** : IBP pour filtrer, CROWN pour affiner

---

<a id="6-applications"></a>

## 6. Applications Avancées

### 6.1 Physics-Informed Neural Networks (PINNs)

**Problème** : Les PINNs doivent respecter des lois physiques (EDOs, EDPs).

**TorchLean permet** :
- Vérifier que les résidus physiques sont bornés
- Garantir la stabilité temporelle
- Certifier la convergence

### 6.2 Contrôle Lyapunov

**Problème** : Garantir la stabilité d'un système de contrôle NN-based.

**Fonction de Lyapunov** : V(x) telle que :
- V(0) = 0
- V(x) > 0 pour x ≠ 0
- dV/dt < 0 (décroissance)

### 6.3 Tableau récapitulatif des applications

| Application | Vérification TorchLean | Bénéfice |
|-------------|------------------------|----------|
| **Classification robuste** | Bornes ε-certifiées | Garantie anti-adversarial |
| **PINNs** | Bornes sur résidus | Convergence physique |
| **Contrôle Lyapunov** | dV/dt < 0 certifié | Stabilité garantie |
| **Médical** | Intervalles de confiance | Sécurité diagnostics |
| **Aérospatial** | Robustesse capteurs | Systèmes critiques |

### 6.4 Intégration avec Python/PyTorch

In [ ]:
-- ===========================================================
-- Exemple 9 : Interoperabilite Lean - Python
-- ===========================================================

-- Type de modele exportable
structure ExportableModel where
  (layers : List Module)
  (verificationResults : Array VerificationResult)
  deriving Repr

-- Metadata pour l'export
structure ExportMetadata where
  (modelName : String)
  (leanVersion : String)
  (torchleanVersion : String)
  (verificationDate : String)
  deriving Repr

-- Exemple : modele verifie a exporter
def verifiedMLP : ExportableModel :=
  { layers := [Module.linear mnistLayer, Module.relu]
    verificationResults := #[ibpResult, crownResult]
  }

def metadata : ExportMetadata :=
  { modelName := "MNIST_Verified"
    leanVersion := "v4.0.0"
    torchleanVersion := "0.1.0"
    verificationDate := "2026-03-03"
  }

#eval "Modele verifiable prepare pour export Python"

### Interprétation : Intégration Lean-Python

**Workflow typique** :

```
1. Python/PyTorch → Entraîner le modèle
                ↓
2. Exporter les poids → Importer dans TorchLean
                ↓
3. Lean 4 → Vérifier formellement
                ↓
4. Exporter certificat → Python pour déploiement
```

**Avantages** :
- Meilleure performance d'entraînement (PyTorch)
- Garanties formelles (TorchLean)
- Déploiement avec certificats vérifiables

---

<a id="7-exercices"></a>

## 7. Exercices

### Exercice 1 : Propagation IBP sur un mini-réseau

Soit un réseau simple : `x → Linear → ReLU → Linear → y`

- Input : `x ∈ [0.5, 1.5]`
- Layer 1 : `weight = 2.0, bias = 0.0`
- Layer 2 : `weight = 1.0, bias = -1.0`

**Question** : Calculez les intervalles après chaque couche.

```lean
-- Votre code ici
def exercise1_input : Interval := { lower := 0.5, upper := 1.5 }

-- TODO: Implementer linearIBP et reluIBP
def exercise1_layer1 : Interval := sorry
def exercise1_relu : Interval := sorry
def exercise1_output : Interval := sorry
```

**Solution** :

| Étape | Calcul | Résultat |
|-------|---------|----------|
| Input | `[0.5, 1.5]` | donné |
| Linear 1 | `[0.5×2, 1.5×2]` | `[1.0, 3.0]` |
| ReLU | `[max(0,1.0), max(0,3.0)]` | `[1.0, 3.0]` |
| Linear 2 | `[1.0×1-1, 3.0×1-1]` | `[0.0, 2.0]` |

---

### Exercice 2 : Certificat de robustesse

Un réseau de classifiction MNIST donne les bornes inférieures suivantes pour `x ∈ [0.48, 0.52]` :

```lean
def exercise2_bounds : Array Float :=
  #[0.1, -0.2, 0.0, 0.3, 0.05, 0.4, 0.0, 2.5, 0.1, 0.2]
-- Classe 0: 0.1, Classe 1: -0.2, ..., Classe 7: 2.5, ...
```

**Questions** :
1. Quelle classe est certifiée robuste ?
2. Quelle est la marge de sécurité (différence minimale) ?
3. Est-ce que ce certificat garantit la robustesse pour `ε = 0.02` ?

```lean
-- Votre code ici
def exercise2_certified_class : Nat := sorry  -- Reponse: 7

def exercise2_margin : Float := sorry  -- Reponse: 2.1 (2.5 - 0.4)

def exercise2_is_robust : Bool := sorry  -- Reponse: true
```

**Solution** :
- Classe certifiée : **7** (score 2.5)
- Deuxième meilleur : **5** (score 0.4)
- Marge : `2.5 - 0.4 = 2.1`
- Robuste pour `ε = 0.02` car la marge est bien supérieure à la perturbation

---

## Exercice 3 : Preuve de propriété Float32

**Théorème à prouver** : L'arrondi RTZ (Round Toward Zero) préserve la positivité stricte.

```lean
theorem roundTz_preserves_positivity (x : Float) :
    (0 < x) → (0 < roundExample RTZ x) := by
  -- TODO: Completer la preuve
  intro h
  unfold roundExample
  -- Indice: utiliser Float.trunc_pos et h
  sorry
```

**Indice** :
1. Dépliez `roundExample` et `RTZ`
2. Utilisez la propriété de `Float.trunc` : `0 < x` implique `0 < trunc(x)` n'est pas toujours vrai!
3. **Correction** : Pour `0 < x < 1`, `trunc(x) = 0`!
4. **Théorème corrigé** : `0 < x` implique `0 ≤ roundTz(x)` (non-strict)

**Version corrigée** :
```lean
theorem roundTz_nonnegative (x : Float) :
    (0 ≤ x) → (0 ≤ roundExample RTZ x) := by
  intro h
  unfold roundExample
  cases x
  all_goals simp [h]
```

---

<a id="8-conclusion"></a>

## 8. Conclusion

### 8.1 Récapitulatif

Dans ce notebook, nous avons vu :

| Concept | Appris | Application |
|---------|--------|-------------|
| **Semantic gap** | Différence PyTorch vs formel | Comprendre les limites de PyTorch |
| **TorchLean** | Architecture 3 modules | Installation et imports |
| **API** | Tenseurs, layers, sequential | Construction de réseaux |
| **Float32** | IEEE-754, arrondi | Preuves d'erreur numérique |
| **IBP** | Propagation d'intervalles | Bornes rapides |
| **CROWN** | Optimisation convexe | Bornes serrées |
| **Applications** | PINNs, Lyapunov | Cas réels |

### 8.2 Tableau comparatif final

| Approche | Vérification | Performance | Scalabilité | Garanties |
|----------|---------------|--------------|-------------|-----------|
| **PyTorch** | Tests empiriques | Maximale | Oui | Statistiques seulement |
| **TorchLean IBP** | Formelle | Bonne | Oui | Conservatrices |
| **TorchLean CROWN** | Formelle | Moyenne | Oui | Serrées |
| **SMT solving** | Formelle | Faible | Non | Exactes mais coûteuses |

### 8.3 Ressources supplémentaires

**TorchLean** :
- Site officiel : [leandojo.org/torchlean.html](https://leandojo.org/torchlean.html)
- GitHub : [github.com/lean-dojo/TorchLean](https://github.com/lean-dojo/TorchLean)

**Vérification de réseaux** :
- IBP original : Gowal et al. (2018)
- CROWN : Singh et al. (2019)
- LiRPA : Xu et al. (2020)

**Lean 4** :
- [Theorem Proving in Lean 4](https://leanprover.github.io/theorem_proving_in_lean4/)
- [Mathematics in Lean](https://leanprover-community.github.io/mathematics_in_lean/)

### 8.4 Prochains pas

Après ce notebook, vous pouvez :

1. **Approfondir TorchLean** : Expérimenter avec des réseaux réels
2. **Lean-12** : Applications avancées de TorchLean (si disponible)
3. **Contribuer** : Au projet TorchLean sur GitHub
4. **Recherche** : Explorer PINNs vérifiés, contrôle formel

### 8.5 Note sur l'état du projet

TorchLean est un projet de recherche actif. Certains composants décrits dans ce notebook peuvent être :
- En cours de développement
- Sujets à modifications API
- Partiellement implémentés

Consultez toujours la documentation officielle pour l'état actuel.

---

**Navigation** : [Index](README.md) | [<< Précédent](Lean-10-LeanDojo.ipynb) | [Version Python](Lean-11-TorchLean-Python.ipynb) |

### 8.6 Synthèse visuelle des concepts TorchLean

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    TORCHLEAN : ECOSYSTEME DE VERIFICATION                   │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌─────────────────┐      ┌─────────────────┐      ┌─────────────────┐    │
│  │   PyTorch       │      │    TorchLean    │      │    Lean 4       │    │
│  │   (Execution)   │ ───▶ │   (Bridge)      │ ───▶ │   (Preuves)     │    │
│  │                 │      │                 │      │                 │    │
│  │ - Training      │      │ - Formalisation │      │ - Theoremes     │    │
│  │ - Inference     │      │ - Semantique    │      │ - Tactics       │    │
│  │ - GPU/TPU       │      │ - Float32       │      │ - Mathlib       │    │
│  └─────────────────┘      └─────────────────┘      └─────────────────┘    │
│           │                        │                        │             │
│           │                        │                        │             │
│           ▼                        ▼                        ▼             │
│  ┌─────────────────────────────────────────────────────────────────┐       │
│                    CERTIFICATS DE ROBUSTESSE                                │
│  ┌─────────────────────────────────────────────────────────────────┐       │
│  │  IBP (Rapide)     │  CROWN (Serré)   │  LiRPA (Complet)      │       │
│  │  ε: 0.01-0.03     │  ε: 0.05-0.10    │  ε: 0.07-0.12         │       │
│  │  Temps: ~0.5s     │  Temps: ~2s       │  Temps: ~5s            │       │
│  └─────────────────────────────────────────────────────────────────┘       │
│                                                                             │
│  APPLICATIONS: PINNs | Controle Lyapunov | Medical | Aeronautique          │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

**Flux de travail typique** :

1. **Développement** (PyTorch) : Prototypage rapide, entraînement GPU
2. **Formalisation** (TorchLean) : Spécification sémantique, imports Lean
3. **Vérification** (Lean 4) : Preuves formelles, certificats de robustesse
4. **Déploiement** (Production) : Code vérifié avec garanties formelles

**Points clés à retenir** :

| Concept | PyTorch | TorchLean |
|---------|---------|-----------|
| Sémantique | Implicite (IEEE-754) | Explicite (formelle) |
| Vérification | Tests empiriques | Preuves formelles |
| Robustesse | Adversarial training | Certificats ε-bornés |
| Garanties | Statistiques | Mathématiques |
| Performance | Maximale (GPU) | Bonne (vérification) |

**Choix méthodologiques** :

- **IBP** : Première ligne de défense, rapide, scalable
- **CROWN** : Certificats de haute qualité, analyse fine
- **LiRPA** : Support étendu d'activations, recherche

> **Note finale** : TorchLean représente une avancée majeure dans la vérification formelle des réseaux de neurones, comblant le fossé entre l'apprentissage profond et les preuves mathématiques rigoureuses.